# Lecția 11 - Protocol agent-la-agent (A2A)


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ce este protocolul A2A?

Protocolul **Agent-la-Agent (A2A)** este un standard deschis care permite agenților AI să comunice,
să se descopere reciproc și să colaboreze — chiar dacă sunt construiți pe cadre diferite sau găzduiți
de servicii diferite.

Concepte cheie:

- **Descoperire** – Agenții publică un *Card al Agentului* care le descrie capacitățile, făcând
  ușor pentru alți agenți (sau orchestratori) să găsească specialistul potrivit pentru o sarcină.
- **Transmiterea mesajelor** – Agenții schimbă mesaje structurate printr-un protocol comun, astfel încât o
  solicitare de la un agent poate fi înțeleasă și îndeplinită de altul indiferent de
  implementarea sa internă.
- **Ciclul de viață al sarcinii** – A2A definește stări precum *submitted*, *working*, *completed*, și
  *failed*, oferind orchestratorului vizibilitate completă asupra modului în care o sarcină delegată progresează.

În această lecție simulăm colaborarea în stil A2A conectând trei agenți de turism specializați
într-un flux de lucru în care fiecare agent își aduce expertiza și transmite rezultatele următorului.


## Crearea agenților de turism specializați


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaborare multi-agent prin flux de lucru

Conectăm cei trei agenți într-un flux de lucru secvențial care reflectă transmiterea de mesaje A2A:

1. **CurrencyExchangeAgent** primește cererea utilizatorului și oferă indicații privind valuta.
2. **ActivityPlannerAgent** primește contextul îmbogățit și adaugă recomandări de activități.
3. **TravelManagerAgent** sintetizează ambele intrări într-un document final de călătorie.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Înțelegerea A2A în mediul de producție

Într-un mediu de producție, protocolul A2A deblochează scenarii puternice între servicii:

| Capacitate | Descriere |
|---|---|
| **Interoperabilitate între framework-uri** | Un agent construit cu un framework poate delega sarcini către un agent construit cu orice alt framework compatibil A2A, permițând interoperabilitatea reală între organizații. |
| **Frontiere ale serviciilor** | Agenții pot exista în microservicii separate, regiuni cloud sau chiar în organizații diferite, continuând să colaboreze fără întreruperi. |
| **Descoperire dinamică** | Un orchestrator poate interoga un registru Agent Card în timpul execuției pentru a găsi specialistul cel mai potrivit pentru o anumită sub-sarcină. |
| **Streaming și notificări push** | A2A acceptă Server-Sent Events (SSE) pentru actualizări de progres în timp real și notificări push pentru sarcini de lungă durată. |

Fluxul de lucru pe care l-am construit mai sus este o versiune simplificată, în cadrul aceluiași proces, a acestui model. Într-o implementare reală
fiecare agent ar expune un endpoint HTTP, ar publica un Agent Card și ar comunica
prin protocolul A2A JSON-RPC.


## Rezumat

În această lecție ai învățat:

1. **Ce este protocolul A2A** — un standard deschis pentru descoperirea între agenți, mesagerie,
   și gestionarea sarcinilor.
2. **Cum să creezi agenți specializați** — un agent pentru schimb valutar, un agent planificator de activități, și un orchestrator Travel Manager.
3. **Cum să conectezi agenții într-un flux de lucru** — folosind `WorkflowBuilder` pentru a modela în mod secvențial
   trimiterea mesajelor între agenți.
4. **Cum funcționează A2A în producție** — permițând colaborarea între framework-uri și servicii cu descoperire dinamică și actualizări în streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm răspunderea pentru eventuale neînțelegeri sau interpretări eronate care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
